# 🧪 A/B Testing & Evaluation: Đánh giá Mô hình Chuẩn hóa / Viết lại Câu hỏi (Question Rewriter)

Notebook này được thiết kế riêng để **so sánh đối đầu (A/B Testing)** giữa:
- **Nhóm A (Baseline):** Câu hỏi gốc (`original_questions`) chứa teencode, viết tắt, không dấu, diễn đạt mơ hồ.
- **Nhóm B (Experiment):** Câu hỏi đã qua mô hình chuẩn hóa / viết lại (`converted_questions`) của đồng đội.

Mục tiêu: Đánh giá xem mô hình viết lại câu hỏi có giúp model Coder (`Qwen2.5-Coder`) sinh SQL/Python chính xác hơn và tăng điểm tổng (Score = Correct - Wrong) hay không.

> 💡 **Mẹo:** Bạn có thể chạy notebook này trực tiếp trên **Google Colab (GPU T4)** hoặc **máy Local** có cài Ollama.

--- 
## 1. Kiểm tra Môi trường & GPU

In [ ]:
# Kiểm tra GPU (Vào Runtime -> Change runtime type -> T4 GPU trước khi chạy)
!nvidia-smi

### 1.1. Thiết lập Mã nguồn & Dữ liệu Dự án (Dành cho Google Colab)
Nếu bạn chạy trên Colab, chỉ cần kéo thả file `Zalo-ai.zip` vào tab Files bên trái rồi chạy cell bên dưới. Cell sẽ tự động giải nén và chuyển thư mục làm việc.

In [ ]:
import os
try:
    import google.colab
    is_colab = True
except ImportError:
    is_colab = False

if is_colab:
    print("⚡ Đang chạy trên Google Colab...")
    if os.path.exists('/content/Zalo-ai.zip') and not os.path.exists('/content/Zalo-ai/src'):
        print("📦 Đang giải nén Zalo-ai.zip...")
        !unzip -q -o /content/Zalo-ai.zip -d /content/Zalo-ai
    if os.path.exists('/content/Zalo-ai'):
        %cd /content/Zalo-ai
        print("📁 Thư mục làm việc hiện tại:", os.getcwd())
    else:
        print("⚠️ Lưu ý: Hãy kéo thả file Zalo-ai.zip vào tab Files (📁) bên trái của Colab rồi chạy lại ô này!")
else:
    print("💻 Đang chạy trên máy cục bộ (Local). Thư mục hiện tại:", os.getcwd())

## 2. Cài đặt các thư viện cần thiết

In [ ]:
!pip install -q pandas duckdb openpyxl pydantic matplotlib seaborn tabulate requests

## 3. Khởi động LLM Backend (Ollama hoặc vLLM)
Mặc định chạy **Ollama** với model Coder (ví dụ: `qwen2.5-coder:7b` hoặc `qwen2.5-coder:3b`).

In [ ]:
import subprocess, os, time

# Thiết lập môi trường GPU
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"
env["OLLAMA_NUM_PARALLEL"] = "4"        # Xử lý song song 4 luồng
env["OLLAMA_FLASH_ATTENTION"] = "1"

# Kiểm tra xem Ollama đã chạy chưa, nếu chưa thì khởi động daemon
try:
    import urllib.request
    urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
    print("✅ Ollama daemon đã sẵn sàng!")
except Exception:
    print("Đang khởi động Ollama daemon...")
    with open("ollama.log", "w") as f:
        subprocess.Popen(["ollama", "serve"], env=env, stdout=f, stderr=f)
    time.sleep(5)
    print("✅ Khởi động Ollama thành công!")

# Tải model nếu chưa có (chọn 7b hoặc 3b)
MODEL_NAME = "qwen2.5-coder:7b"  # Hoặc 'qwen2.5-coder:3b'
!ollama pull {MODEL_NAME}
!ollama ps

--- 
## 4. Cấu hình File Câu hỏi Đã Convert (Input Selection)

Định dạng file câu hỏi đã convert (`.jsonl`):
```json
{"id": "q_001883280e83", "business_id": "biz_128", "question": "câu hỏi sau khi viết lại...", "as_of_date": "2026-09-07"}
```

*(Nếu bạn chưa nhận được file từ bạn cùng nhóm, notebook đã chuẩn bị sẵn file mẫu `notebooks/sample_converted_questions.jsonl` với 10 câu mẫu để bạn chạy thử nghiệm ngay!)*

In [ ]:
from pathlib import Path
import json

# Đường dẫn tới file câu hỏi đã convert do bạn cùng nhóm cung cấp:
# Bạn có thể thay đổi đường dẫn này sang file bạn tải lên (ví dụ: 'converted_questions.jsonl')
CONVERTED_FILE = "notebooks/sample_converted_questions.jsonl"
SPLIT_DIR = "train"

# Số lượng câu muốn test thử (đặt None để test toàn bộ file)
TEST_LIMIT = 10

assert Path(CONVERTED_FILE).exists(), f"Không tìm thấy file: {CONVERTED_FILE}"

# Đọc xem trước 3 câu đầu tiên
print(f"📂 Đang sử dụng file: {CONVERTED_FILE}")
with open(CONVERTED_FILE, encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        item = json.loads(line)
        print(f"\n[{i+1}] ID: {item.get('id')}")
        if 'original_question' in item:
            print(f"   - Câu gốc : {item.get('original_question')}")
        print(f"   - Câu sửa : {item.get('question')}")

--- 
## 5. Thực thi Kiểm thử A/B Song Song

Chúng ta sẽ thực hiện 2 lần chạy với cùng điều kiện:
1. **Baseline Run (A):** Chạy trên câu hỏi gốc (`train/questions.jsonl`).
2. **Experiment Run (B):** Chạy trên câu hỏi đã qua mô hình sửa đổi (`CONVERTED_FILE`).

In [ ]:
# Tạo thư mục chứa kết quả A/B Test
os.makedirs("result/ab_test", exist_ok=True)

limit_flag = f"--limit {TEST_LIMIT}" if TEST_LIMIT else ""

print("=" * 60)
print("🚀 [PHASE A] CHẠY BASELINE VỚI CÂU HỎI GỐC...")
print("=" * 60)
!python scripts/run_benchmark.py \
  --split-dir {SPLIT_DIR} \
  --model-backend ollama \
  --model-name {MODEL_NAME} \
  --workers 4 \
  {limit_flag} \
  --output result/ab_test/submission_baseline.jsonl \
  --log-dir result/ab_test/logs_baseline \
  --evaluate

print("\n" + "=" * 60)
print("✨ [PHASE B] CHẠY EXPERIMENT VỚI CÂU HỎI ĐÃ CONVERT...")
print("=" * 60)
!python scripts/run_benchmark.py \
  --split-dir {SPLIT_DIR} \
  --questions-file {CONVERTED_FILE} \
  --model-backend ollama \
  --model-name {MODEL_NAME} \
  --workers 4 \
  {limit_flag} \
  --output result/ab_test/submission_converted.jsonl \
  --log-dir result/ab_test/logs_converted \
  --evaluate

--- 
## 6. Phân tích & So sánh Đối chiếu (A/B Comparative Evaluation)

Tải kết quả dự đoán của 2 nhóm và chấm điểm chi tiết từng câu theo công thức Zalo AI:
- Đúng: **+1 điểm**
- Sai: **-1 điểm**
- Bỏ qua (Abstain): **0 điểm**

In [ ]:
import pandas as pd
from src.evaluator import Evaluator

def load_jsonl(path):
    items = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            if line.strip():
                items.append(json.loads(line))
    return items

answers_gt = {a['id']: a for a in load_jsonl(f"{SPLIT_DIR}/answers.jsonl")}
base_preds = {p['id']: p for p in load_jsonl("result/ab_test/submission_baseline.jsonl")}
conv_preds = {p['id']: p for p in load_jsonl("result/ab_test/submission_converted.jsonl")}
orig_qs = {q['id']: q for q in load_jsonl(f"{SPLIT_DIR}/questions.jsonl")}
conv_qs = {q['id']: q for q in load_jsonl(CONVERTED_FILE)}

# Đánh giá metrics tổng thể
all_ids = sorted(list(set(base_preds.keys()) & set(conv_preds.keys())))
eval_base = Evaluator.evaluate([base_preds[i] for i in all_ids], [answers_gt[i] for i in all_ids])
eval_conv = Evaluator.evaluate([conv_preds[i] for i in all_ids], [answers_gt[i] for i in all_ids])

summary_df = pd.DataFrame({
    "Tiêu chí": ["Tổng số câu đánh giá", "Số câu Đúng (+1)", "Số câu Sai (-1)", "Số câu Abstain (0)", "TỔNG ĐIỂM (Score)", "Độ chính xác (Accuracy %)"],
    "Câu hỏi Gốc (Baseline)": [
        eval_base["total_questions"], eval_base["correct"], eval_base["wrong"], eval_base["abstain"],
        eval_base["total_score"], f"{eval_base['accuracy']:.2f}%"
    ],
    "Câu hỏi Đã Sửa (Converted)": [
        eval_conv["total_questions"], eval_conv["correct"], eval_conv["wrong"], eval_conv["abstain"],
        eval_conv["total_score"], f"{eval_conv['accuracy']:.2f}%"
    ],
    "Mức cải thiện (Δ Delta)": [
        0,
        f"{eval_conv['correct'] - eval_base['correct']:+d}",
        f"{eval_conv['wrong'] - eval_base['wrong']:+d}",
        f"{eval_conv['abstain'] - eval_base['abstain']:+d}",
        f"{eval_conv['total_score'] - eval_base['total_score']:+d} điểm",
        f"{eval_conv['accuracy'] - eval_base['accuracy']:+.2f}%"
    ]
})

print("📊 BẢNG TỔNG KẾT HIỆU QUẢ A/B TESTING:")
display(summary_df)

--- 
## 7. Phân tích Chi tiết: Các Câu Thay Đổi Kết Quả (Wins & Losses)

- 🟢 **Win (Cải thiện):** Gốc Sai/Abstain $\rightarrow$ Sau khi sửa câu hỏi thì Coder làm **Đúng** (+2 điểm).
- 🔴 **Loss (Thụt lùi):** Gốc Đúng $\rightarrow$ Sau khi sửa câu hỏi lại bị **Sai** (-2 điểm). *Đây là các trường hợp cực kỳ quan trọng cần gửi cho đồng đội để fine-tune lại model rewriter!*

In [ ]:
records = []

for q_id in all_ids:
    gt = answers_gt[q_id]
    b_res = Evaluator.evaluate([base_preds[q_id]], [gt])
    c_res = Evaluator.evaluate([conv_preds[q_id]], [gt])
    
    b_status = "ĐÚNG" if b_res["correct"] == 1 else ("ABSTAIN" if b_res["abstain"] == 1 else "SAI")
    c_status = "ĐÚNG" if c_res["correct"] == 1 else ("ABSTAIN" if c_res["abstain"] == 1 else "SAI")
    
    # Xác định trạng thái tác động
    if b_status != "ĐÚNG" and c_status == "ĐÚNG":
        impact = "🟢 CẢI THIỆN (WIN)"
    elif b_status == "ĐÚNG" and c_status != "ĐÚNG":
        impact = "🔴 THỤT LÙI (LOSS)"
    elif b_status == "SAI" and c_status == "ABSTAIN":
        impact = "🟡 GIẢM THIỆT HẠI (Saved -1)"
    else:
        impact = "⚪ KHÔNG ĐỔI"
        
    records.append({
        "ID": q_id,
        "Shop": orig_qs[q_id].get("business_id"),
        "Tác động": impact,
        "Gốc": b_status,
        "Sửa": c_status,
        "Đáp án GT": gt.get("answer", gt.get("reason")),
        "Pred Gốc": base_preds[q_id].get("answer", base_preds[q_id].get("reason")),
        "Pred Sửa": conv_preds[q_id].get("answer", conv_preds[q_id].get("reason")),
        "Câu hỏi Gốc": orig_qs[q_id]["question"],
        "Câu hỏi Đã Sửa": conv_qs[q_id]["question"]
    })

detail_df = pd.DataFrame(records)

# Lọc ra các câu có sự thay đổi
changed_df = detail_df[detail_df["Tác động"] != "⚪ KHÔNG ĐỔI"]
print(f"🔍 Tìm thấy {len(changed_df)} / {len(detail_df)} câu có sự thay đổi kết quả:")
display(changed_df[["ID", "Tác động", "Gốc", "Sửa", "Câu hỏi Gốc", "Câu hỏi Đã Sửa"]])

--- 
## 8. Biểu đồ Trực quan hóa So sánh (Visual Dashboard)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: So sánh Điểm tổng và Số câu đúng
metrics_names = ["Tổng Điểm (Score)", "Số câu Đúng", "Số câu Sai"]
base_vals = [eval_base["total_score"], eval_base["correct"], eval_base["wrong"]]
conv_vals = [eval_conv["total_score"], eval_conv["correct"], eval_conv["wrong"]]

x = range(len(metrics_names))
width = 0.35

axes[0].bar([i - width/2 for i in x], base_vals, width, label="Câu hỏi Gốc (Baseline)", color="#95a5a6")
axes[0].bar([i + width/2 for i in x], conv_vals, width, label="Câu hỏi Đã Sửa (Converted)", color="#2ecc71")
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(metrics_names, fontsize=11)
axes[0].set_title("So sánh Điểm số & Số câu Đúng / Sai", fontsize=13, fontweight='bold')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)
axes[0].legend()

# Biểu đồ 2: Cơ cấu Tác động của Model Rewriter
impact_counts = detail_df["Tác động"].value_counts()
colors = ["#2ecc71" if "CẢI THIỆN" in k else ("#e74c3c" if "THỤT LÙI" in k else ("#f39c12" if "GIẢM" in k else "#bdc3c7")) for k in impact_counts.index]

axes[1].pie(impact_counts, labels=impact_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=[0.05]*len(impact_counts))
axes[1].set_title("Cơ cấu Tác động của Model Viết lại Câu hỏi", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

--- 
## 9. Xuất Báo cáo CSV gửi cho Bạn Cùng Nhóm
File CSV này chứa chi tiết từng câu hỏi, kèm nhãn `Tác động` (Cải thiện / Thụt lùi) và lý do sai, giúp bạn cùng nhóm biết chính xác model rewriter cần cải thiện ở những điểm nào.

In [ ]:
report_path = "result/ab_test/ab_test_comparison_report.csv"
detail_df.to_csv(report_path, index=False, encoding='utf-8-sig')
print(f"✅ Đã xuất báo cáo đối chiếu chi tiết ra file: {report_path}")

# Nếu đang chạy trên Colab, tự động tải file về máy:
try:
    from google.colab import files
    files.download(report_path)
except Exception:
    pass